# Q2: Distance Matrix Computation

This notebook uses `lab2 data.csv` to demonstrate distance matrix computation with two methods:

1. Euclidean distance
2. Manhattan distance

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

file_path = 'lab2 data.csv'
df = pd.read_csv(file_path)
df.head()

,Timestamp,Reg No,Job role that you are interested in,What is the minimum salary of students placed through campus (In LPA..respond as a number),What is the maximum salary of students placed through campus (In LPA..respond as a number),What is the median salary of students placed through campus (In LPA..respond as a number),Which is the highest paying company,Rate your contribution towards extra curricular activities,Rate your technical competencies,What are your package expectations (LPA),your CIA % of last semester,your GPA of last semester,Your maximum attendance % till last semester,Internships Interests
0,12-4-2024 20:31:09,2341338,Data Scientist,3 LPA,1Cr,20 LPA,Deolite,3.0,4,20 LPA,60%,62%,77%,Industry
1,12-4-2024 20:33:07,2341310,Data Scientist,5,NaN,NaN,DE Shaw,3.0,2,11,67,60,97,Industry
2,12-4-2024 20:39:43,2341311,Data Scientist,7 LPA,15 LPA,8 LPA,DE Shaw,3.0,4,5 LPA,80,80,99,Research
3,12-4-2024 20:40:37,2341324,Data Scientist,3 LPA,25 LPA,10 LPA,DE Shaw,NaN,3,20,75,70,95,Research
4,12-4-2024 20:41:51,2341324,Business Analyst,2,50,25,Deolite,3.0,3,20,65,76,96,Industry


In [7]:
# Select useful numeric-like columns from the survey data.
selected_columns = [
    'What is the minimum salary of students placed through campus (In LPA..respond as a number)',
    'What is the maximum salary of students placed through campus (In LPA..respond as a number)',
    'What is the  median salary of students placed through campus (In LPA..respond as a number)',
    'Rate your contribution towards extra curricular activities',
    'Rate your technical competencies',
    'What are your package expectations (LPA)',
    'your CIA % of last semester',
    'your GPA of last semester',
    'Your maximum attendance % till last semester'
]

numeric_df = df[selected_columns].copy()

salary_columns = {
    'What is the minimum salary of students placed through campus (In LPA..respond as a number)',
    'What is the maximum salary of students placed through campus (In LPA..respond as a number)',
    'What is the  median salary of students placed through campus (In LPA..respond as a number)',
    'What are your package expectations (LPA)'
}

def clean_numeric_value(value, is_salary=False):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()
    if text == '':
        return np.nan

    if is_salary and any(ch.isalpha() for ch in text):
        valid_salary_terms = ('lpa', 'lac', 'lakh', 'lakhs', 'cr', 'crore')
        if not any(term in text for term in valid_salary_terms):
            return np.nan

    cleaned_text = ''.join(ch for ch in text if ch.isdigit() or ch in {'.', ','})
    if cleaned_text == '':
        return np.nan

    number = float(cleaned_text.replace(',', ''))

    if is_salary:
        if 'cr' in text or 'crore' in text:
            return number * 100.0
        if ',' in text or number > 1000:
            return number / 100000.0
        return number

    return number

for column in numeric_df.columns:
    numeric_df[column] = numeric_df[column].apply(
        lambda value: clean_numeric_value(value, is_salary=column in salary_columns)
    )

# Fill missing values with column mean so distance calculation can run smoothly.
numeric_df = numeric_df.fillna(numeric_df.mean(numeric_only=True))

numeric_df.head()

,What is the minimum salary of students placed through campus (In LPA..respond as a number),What is the maximum salary of students placed through campus (In LPA..respond as a number),What is the median salary of students placed through campus (In LPA..respond as a number),Rate your contribution towards extra curricular activities,Rate your technical competencies,What are your package expectations (LPA),your CIA % of last semester,your GPA of last semester,Your maximum attendance % till last semester
0,3.0,100.000000,20.000000,3.000000,4.0,20.0,60.0,62.0,77.0
1,5.0,37.556987,13.182786,3.000000,2.0,11.0,67.0,60.0,97.0
2,7.0,15.000000,8.000000,3.000000,4.0,5.0,80.0,80.0,99.0
3,3.0,25.000000,10.000000,3.382716,3.0,20.0,75.0,70.0,95.0
4,2.0,50.000000,25.000000,3.000000,3.0,20.0,65.0,76.0,96.0


In [8]:
# Use the first 10 records to keep the matrix readable in the notebook.
sample_df = numeric_df.head(10).copy()
sample_df.index = [f'Student_{i+1}' for i in range(len(sample_df))]
sample_df

,What is the minimum salary of students placed through campus (In LPA..respond as a number),What is the maximum salary of students placed through campus (In LPA..respond as a number),What is the median salary of students placed through campus (In LPA..respond as a number),Rate your contribution towards extra curricular activities,Rate your technical competencies,What are your package expectations (LPA),your CIA % of last semester,your GPA of last semester,Your maximum attendance % till last semester
Student_1,3.0,100.000000,20.000000,3.000000,4.0,20.0,60.0,62.0,77.0
Student_2,5.0,37.556987,13.182786,3.000000,2.0,11.0,67.0,60.0,97.0
Student_3,7.0,15.000000,8.000000,3.000000,4.0,5.0,80.0,80.0,99.0
Student_4,3.0,25.000000,10.000000,3.382716,3.0,20.0,75.0,70.0,95.0
Student_5,2.0,50.000000,25.000000,3.000000,3.0,20.0,65.0,76.0,96.0
Student_6,3.0,30.000000,20.000000,3.000000,2.0,40.0,79.0,69.0,98.0
Student_7,2.0,71.000000,22.000000,5.000000,5.0,20.0,68.0,80.0,99.0
Student_8,3.0,77.000000,25.000000,4.000000,4.0,25.0,75.0,87.0,97.0
Student_9,3.0,24.000000,6.000000,5.000000,5.0,20.0,65.0,70.0,98.0
Student_10,2.0,15.000000,8.000000,4.000000,4.0,8.0,60.0,75.0,95.0


In [9]:
def euclidean_distance_matrix(dataframe):
    data = dataframe.to_numpy(dtype=float)
    diff = data[:, np.newaxis, :] - data[np.newaxis, :, :]
    distances = np.sqrt(np.sum(diff ** 2, axis=2))
    return pd.DataFrame(distances, index=dataframe.index, columns=dataframe.index)

def manhattan_distance_matrix(dataframe):
    data = dataframe.to_numpy(dtype=float)
    diff = np.abs(data[:, np.newaxis, :] - data[np.newaxis, :, :])
    distances = np.sum(diff, axis=2)
    return pd.DataFrame(distances, index=dataframe.index, columns=dataframe.index)

euclidean_matrix = euclidean_distance_matrix(sample_df)
manhattan_matrix = manhattan_distance_matrix(sample_df)

In [ ]:
# Graphical comparison of the two distance matrices.
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(euclidean_matrix, annot=True, fmt='.1f', cmap='Blues', ax=axes[0])
axes[0].set_title('Euclidean Distance Matrix')
axes[0].set_xlabel('Students')
axes[0].set_ylabel('Students')

sns.heatmap(manhattan_matrix, annot=True, fmt='.1f', cmap='Oranges', ax=axes[1])
axes[1].set_title('Manhattan Distance Matrix')
axes[1].set_xlabel('Students')
axes[1].set_ylabel('Students')

plt.tight_layout()
plt.show()

In [10]:
print('Euclidean Distance Matrix:')
display(euclidean_matrix.round(2))

print('Manhattan Distance Matrix:')
display(manhattan_matrix.round(2))

Euclidean Distance Matrix:


,Student_1,Student_2,Student_3,Student_4,Student_5,Student_6,Student_7,Student_8,Student_9,Student_10
Student_1,0.00,66.99,93.90,79.62,55.76,78.45,41.51,42.78,80.67,89.49
Student_2,66.99,0.00,33.95,20.54,25.43,34.27,41.28,51.90,20.94,28.93
Student_3,93.90,33.95,0.00,22.07,44.89,41.68,61.08,68.03,25.63,21.82
Student_4,79.62,20.54,22.07,0.00,31.43,23.50,49.32,56.99,11.52,22.37
Student_5,55.76,25.43,44.89,31.43,0.00,32.80,22.18,31.29,32.95,41.07
Student_6,78.45,34.27,41.68,23.50,32.80,0.00,48.39,52.96,29.02,42.49
Student_7,41.51,41.28,61.08,49.32,22.18,48.39,0.00,13.23,50.75,59.86
Student_8,42.78,51.90,68.03,56.99,31.29,52.96,13.23,0.00,59.89,69.25
Student_9,80.67,20.94,25.63,11.52,32.95,29.02,50.75,59.89,0.00,17.06
Student_10,89.49,28.93,21.82,22.37,41.07,42.49,59.86,69.25,17.06,0.00


Manhattan Distance Matrix:


,Student_1,Student_2,Student_3,Student_4,Student_5,Student_6,Student_7,Student_8,Student_9,Student_10
Student_1,0.00,111.26,176.00,127.38,95.00,139.00,83.00,94.00,127.00,142.00
Student_2,111.26,0.00,72.74,48.12,56.26,67.37,82.26,105.26,49.74,60.74
Student_3,176.00,72.74,0.00,51.38,95.00,81.00,105.00,118.00,59.00,38.00
Student_4,127.38,48.12,51.38,0.00,58.38,44.38,83.62,92.62,21.62,46.62
Student_5,95.00,56.26,95.00,58.38,0.00,70.00,38.00,57.00,58.00,73.00
Student_6,139.00,67.37,81.00,44.38,70.00,0.00,92.00,93.00,60.00,91.00
Student_7,83.00,82.26,105.00,83.62,38.00,92.00,0.00,33.00,78.00,101.00
Student_8,94.00,105.26,118.00,92.62,57.00,93.00,33.00,0.00,107.00,126.00
Student_9,127.00,49.74,59.00,21.62,58.00,60.00,78.00,107.00,0.00,39.00
Student_10,142.00,60.74,38.00,46.62,73.00,91.00,101.00,126.00,39.00,0.00


## Conclusion

- The **Euclidean distance matrix** measures straight-line distance between students based on the selected numeric features.
- The **Manhattan distance matrix** measures city-block distance by summing absolute differences across features.
- Both methods help compare similarity or dissimilarity between records in the dataset.